# 🧠 Brain Tumor Segmentation on BRISC2025 (U-Net)

**Author:** Md Sagor Hossain

Trains a U-Net (ResNet34 encoder, ImageNet-pretrained) for **binary brain tumor segmentation** on the BRISC2025 dataset. Designed to run directly on **Kaggle** with the dataset attached as an input.

**Dataset path used in this notebook:**
`/kaggle/input/datasets/briscdataset/brisc2025/brisc2025`

Expected structure:
```
brisc2025/
└── segmentation_task/
    ├── train/
    │   ├── images/*.jpg
    │   └── masks/*.png
    └── test/
        ├── images/*.jpg
        └── masks/*.png
```

## 1. Setup

In [ ]:
!pip install segmentation-models-pytorch albumentations -q

In [ ]:
import os
import glob
import json

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_ROOT = "/kaggle/input/datasets/briscdataset/brisc2025/brisc2025"
IMG_SIZE = 256

## 2. Dataset Class

In [ ]:
class BriscSegDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.images_dir = os.path.join(root_dir, "segmentation_task", split, "images")
        self.masks_dir = os.path.join(root_dir, "segmentation_task", split, "masks")
        self.image_paths = sorted(glob.glob(os.path.join(self.images_dir, "*.jpg")))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def _mask_path_for(self, image_path):
        basename = os.path.splitext(os.path.basename(image_path))[0]
        return os.path.join(self.masks_dir, basename + ".png")

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        mask_path = self._mask_path_for(image_path)

        image = np.array(Image.open(image_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"))

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        mask = (mask > 0).float().unsqueeze(0)
        return image, mask

## 3. Transforms & DataLoaders

In [ ]:
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
test_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_dataset = BriscSegDataset(DATA_ROOT, split="train", transform=train_tf)
test_dataset = BriscSegDataset(DATA_ROOT, split="test", transform=test_tf)

print("Train samples:", len(train_dataset), "| Test samples:", len(test_dataset))

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 4. Visualize a Sample (Image + Mask)

In [ ]:
image, mask = train_dataset[0]
img_show = image.permute(1, 2, 0).numpy()
img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min())

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_show); axes[0].set_title("MRI Image"); axes[0].axis("off")
axes[1].imshow(mask[0].numpy(), cmap="gray"); axes[1].set_title("Tumor Mask"); axes[1].axis("off")
plt.tight_layout()
plt.show()

## 5. Model — U-Net with ResNet34 Encoder (ImageNet-pretrained)

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(device)
print("Model loaded.")

## 6. Loss & Metrics — Dice + BCE, Dice/IoU scoring

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight

    def forward(self, logits, target):
        bce_loss = self.bce(logits, target)
        probs = torch.sigmoid(logits)
        intersection = (probs * target).sum(dim=(1, 2, 3))
        union = probs.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        dice_loss = 1 - ((2 * intersection + 1e-6) / (union + 1e-6)).mean()
        return self.bce_weight * bce_loss + (1 - self.bce_weight) * dice_loss


def dice_coefficient(pred, target, eps=1e-6):
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return ((2 * intersection + eps) / (union + eps)).mean().item()


def iou_score(pred, target, eps=1e-6):
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) - intersection
    return ((intersection + eps) / (union + eps)).mean().item()


criterion = DiceBCELoss(bce_weight=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

## 7. Training Loop

In [ ]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_dice, total_iou, n = 0.0, 0.0, 0.0, 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            if is_train:
                optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, masks)
            if is_train:
                loss.backward()
                optimizer.step()
            probs = torch.sigmoid(logits)
            total_loss += loss.item()
            total_dice += dice_coefficient(probs, masks)
            total_iou += iou_score(probs, masks)
            n += 1
    return total_loss / n, total_dice / n, total_iou / n

In [ ]:
EPOCHS = 20
history = {"train_loss": [], "val_loss": [], "train_dice": [], "val_dice": []}
best_val_dice = 0.0
os.makedirs("models", exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_dice, train_iou = run_epoch(model, train_loader, criterion, device, optimizer)
    val_loss, val_dice, val_iou = run_epoch(model, test_loader, criterion, device)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_dice"].append(train_dice)
    history["val_dice"].append(val_dice)

    print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} Dice: {train_dice:.4f} IoU: {train_iou:.4f} "
          f"| Val Loss: {val_loss:.4f} Dice: {val_dice:.4f} IoU: {val_iou:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), "models/unet_brisc_seg.pth")

print(f"\nBest validation Dice: {best_val_dice:.4f}")

## 8. Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")
axes[0].plot(epochs_range, history["val_loss"], marker="o", label="Val Loss")
axes[0].set_title("Loss per Epoch"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_dice"], marker="o", label="Train Dice")
axes[1].plot(epochs_range, history["val_dice"], marker="o", label="Val Dice")
axes[1].set_title("Dice Score per Epoch"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/loss_dice_curves.png", dpi=150)
plt.show()

## 9. Qualitative Results — Sample Predictions

In [ ]:
model.load_state_dict(torch.load("models/unet_brisc_seg.pth", map_location=device))
model.eval()

images, masks = next(iter(test_loader))
with torch.no_grad():
    preds = torch.sigmoid(model(images.to(device))).cpu()

n = 4
fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
for i in range(n):
    img = images[i].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    gt = masks[i, 0].numpy()
    pred = (preds[i, 0].numpy() > 0.5).astype(float)

    axes[i, 0].imshow(img); axes[i, 0].set_title("MRI Image"); axes[i, 0].axis("off")
    axes[i, 1].imshow(gt, cmap="gray"); axes[i, 1].set_title("Ground Truth"); axes[i, 1].axis("off")
    axes[i, 2].imshow(pred, cmap="gray"); axes[i, 2].set_title("Prediction"); axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig("results/sample_predictions.png", dpi=150)
plt.show()

## 10. Final Test Set Evaluation

In [ ]:
final_loss, final_dice, final_iou = run_epoch(model, test_loader, criterion, device)
print(f"Final Test Loss: {final_loss:.4f}")
print(f"Final Test Dice Score: {final_dice:.4f}")
print(f"Final Test IoU: {final_iou:.4f}")

with open("results/final_metrics.json", "w") as f:
    json.dump({"test_loss": final_loss, "test_dice": final_dice, "test_iou": final_iou}, f, indent=2)

## 11. Conclusion

- A U-Net with a pretrained ResNet34 encoder was fine-tuned for binary tumor segmentation on BRISC2025.
- Combined Dice + BCE loss handles the class imbalance between tumor and background pixels.
- Training/validation Dice and IoU curves indicate whether the model is converging well or overfitting (check the plot above).
- Model weights are saved to `models/unet_brisc_seg.pth` for reuse via `src/predict.py`.

**Future Work:**
- Try other encoders (efficientnet-b3, resnext50) or architectures (U-Net++, DeepLabV3+)
- Multi-class segmentation (glioma vs. meningioma vs. pituitary boundaries) instead of binary
- Combine with the classification model for a multi-task pipeline (classify + localize)